# Search4Cure.AI: executable walkthrough

Run the cells in order. This notebook calls the project functions to query **real arXiv data**, download one version, and extract text by physical PDF page. Running it again makes new source requests and creates new local retrieval/extraction records.

Use the **Search4Cure (.venv)** kernel after the notebook setup in the README. Saved outputs are snapshots of a particular execution, not evidence that the latest source results are unchanged.

In [1]:
import json
import logging
import os
from pathlib import Path
import ssl
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "arxiv_search.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "arxiv_search.py").is_file():
    raise RuntimeError("Open the notebook from the Search4Cure project directory.")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Use an existing system trust store only when this macOS Python lacks a default CA file.
if sys.platform == "darwin" and ssl.get_default_verify_paths().cafile is None:
    system_ca = Path("/etc/ssl/cert.pem")
    if system_ca.is_file():
        os.environ["SSL_CERT_FILE"] = str(system_ca)
        print("Using the system CA bundle; HTTPS verification remains enabled.")

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)

import pymupdf
from arxiv_search import search_one
from download_pdf import download_pdf
from extract_text import extract_text

QUERY = "all:diabetes"
# The working directory is the project root, including when opened from notebooks/.
OUTPUT_DIR = Path("data/arxiv")
print(f"Python {sys.version.split()[0]}; PyMuPDF {pymupdf.VersionBind}")
print(f"Search query: {QUERY}")

Using the system CA bundle; HTTPS verification remains enabled.
Python 3.13.12; PyMuPDF 1.28.2
Search query: all:diabetes


## 1. Query publication metadata

This cell contacts the arXiv API for at most one result. Keep first-submission time, version-submission time, and our retrieval time distinct. Unknown fields remain unknown.

In [2]:
print("Querying arXiv...")
search_result = search_one(QUERY)
if search_result["document"] is None:
    raise RuntimeError("No publication matched this query.")
print(json.dumps(search_result, indent=2, ensure_ascii=False))

Querying arXiv...


{
  "query": "all:diabetes",
  "request_url": "https://export.arxiv.org/api/query?search_query=all%3Adiabetes&start=0&max_results=1&sortBy=submittedDate&sortOrder=descending",
  "retrieved_at": "2026-11-04T12:47:39.557555+00:00",
  "document": {
    "arxiv_id": "2611.04405",
    "version": 1,
    "title": "DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for Imbalanced Diabetic Retinopathy Grading via Adaptive SMOTE Fusion",
    "source_url": "http://arxiv.org/abs/2611.04405v1",
    "first_submitted_at": "2026-11-03T06:44:25Z",
    "version_submitted_at": "2026-11-03T06:44:25Z"
  }
}


## 2. Download the exact version

The download requires a known version, streams bytes to disk, and records a SHA-256 fingerprint and a separate PDF retrieval time. The pause respects the API request spacing. Only one importer should run at a time.

In [3]:
print("Waiting 3.1 seconds before the PDF request...")
time.sleep(3.1)
download_result = download_pdf(search_result, OUTPUT_DIR)
print(json.dumps(download_result, indent=2, ensure_ascii=False))

Waiting 3.1 seconds before the PDF request...


PDF saved: data/arxiv/2611.04405v1/932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075.pdf (577377 bytes)


Metadata saved: data/arxiv/2611.04405v1/0b1e0c52ca074a38a125f14e4fa2fcfc.json


{
  "search": {
    "query": "all:diabetes",
    "request_url": "https://export.arxiv.org/api/query?search_query=all%3Adiabetes&start=0&max_results=1&sortBy=submittedDate&sortOrder=descending",
    "retrieved_at": "2026-11-04T12:47:39.557555+00:00",
    "document": {
      "arxiv_id": "2611.04405",
      "version": 1,
      "title": "DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for Imbalanced Diabetic Retinopathy Grading via Adaptive SMOTE Fusion",
      "source_url": "http://arxiv.org/abs/2611.04405v1",
      "first_submitted_at": "2026-11-03T06:44:25Z",
      "version_submitted_at": "2026-11-03T06:44:25Z"
    }
  },
  "pdf_url": "https://arxiv.org/pdf/2611.04405v1",
  "pdf_retrieved_at": "2026-11-04T12:47:42.881810+00:00",
  "sha256": "932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075",
  "size_bytes": 577377,
  "local_path": "data/arxiv/2611.04405v1/932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075.pdf",
  "status": "downloaded",
  "metada

## 3. Extract text by physical PDF page

The extractor verifies the saved PDF hash before parsing it. It records the parser version/configuration, page numbers starting at 1, empty pages, and any NUL replacements. Each attempt creates its own JSON result; a failed attempt does not replace the source or a previous result.

In [4]:
extraction_result = extract_text(download_result["metadata_path"])
assert extraction_result["download"]["sha256"] == download_result["sha256"]
assert extraction_result["page_count"] == len(extraction_result["pages"])
summary = {
    "arxiv_id": extraction_result["download"]["search"]["document"]["arxiv_id"],
    "version": extraction_result["download"]["search"]["document"]["version"],
    "status": extraction_result["status"],
    "page_count": extraction_result["page_count"],
    "empty_pages": extraction_result["empty_pages"],
    "total_characters": extraction_result["total_characters"],
    "parser": extraction_result["parser"],
    "parser_repaired_pdf": extraction_result["parser_repaired_pdf"],
    "output": extraction_result["metadata_path"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

Verifying PDF hash: data/arxiv/2611.04405v1/932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075.pdf


Page 1/6: 6714 characters


Page 2/6: 7058 characters


Page 3/6: 4269 characters


Page 4/6: 7055 characters


Page 5/6: 6945 characters


Page 6/6: 5573 characters


Extraction record saved: data/arxiv/2611.04405v1/extracted/d347afbb02a7460c9cc56cf16fd079fc.json


{
  "arxiv_id": "2611.04405",
  "version": 1,
  "status": "extracted",
  "page_count": 6,
  "empty_pages": [],
  "total_characters": 37614,
  "parser": {
    "name": "PyMuPDF",
    "version": "1.28.2",
    "text_mode": "text",
    "sort": true,
    "ocr": false,
    "page_numbering": "physical PDF pages, starting at 1",
    "nul_handling": "replace NUL with U+FFFD and record the replacement count"
  },
  "parser_repaired_pdf": false,
  "output": "data/arxiv/2611.04405v1/extracted/d347afbb02a7460c9cc56cf16fd079fc.json"
}


## 4. Inspect one page

Change `PAGE_NUMBER` and rerun only this cell to inspect a different page. These are physical PDF pages, which may differ from printed page labels. A nonempty extraction does not by itself prove correct reading order or scientific relevance.

In [5]:
PAGE_NUMBER = 1
page = extraction_result["pages"][PAGE_NUMBER - 1]
document = extraction_result["download"]["search"]["document"]
print(f"Source: {document['arxiv_id']}v{document['version']}, physical page {page['page_number']}")
print(f"Source URL: {document['source_url']}")
print(f"PDF SHA-256: {extraction_result['download']['sha256']}")
print(f"Page status: {page['status']}; NUL replacements: {page['nul_replacements']}")
print("\nText preview (first 500 characters):\n")
print(page["text"][:500])

Source: 2611.04405v1, physical page 1
Source URL: http://arxiv.org/abs/2611.04405v1
PDF SHA-256: 932ec59803c4c1495ebfbb7dd3b58e17afdbbc24e459e5a48f35f48a54bd3075
Page status: extracted; NUL replacements: 0

Text preview (first 500 characters):

    DR-LiteNet: A Lightweight Explainable Hybrid CNN Framework for
    Imbalanced Diabetic Retinopathy Grading via Adaptive SMOTE Fusion


                                Nishi Kanta Paul    Md Shihabul Islam Shovo
                                 NIMISHES Lab               NIMISHES Lab

                                        Dhaka, Bangladesh                Dhaka, Bangladesh

                                    nishikantapaul108@gmail.com        shihabul900@gmail.com

                         


## 5. Run the offline tests

These tests use synthetic API responses and generated PDFs, not the publication above. Expected synthetic network-failure warnings exercise error handling. The final unittest result determines success.

In [6]:
command = [
    sys.executable, "-u", "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_*.py", "-v",
]
print("Running offline tests...", flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    exit_code = process.wait()
print(f"Test process exit code: {exit_code}")
if exit_code != 0:
    raise RuntimeError("The offline tests failed.")

Running offline tests...


test_empty_and_invalid_entries (test_arxiv_search.ArxivSearchTests.test_empty_and_invalid_entries) ... ok


test_rate_limit_and_retry_after_stop_automatic_retries (test_arxiv_search.ArxivSearchTests.test_rate_limit_and_retry_after_stop_automatic_retries) ... WARNING:root:arXiv request failed: attempt=1/3 error=HTTP Error 429: Synthetic HTTP error


ok


test_real_request_shape_limits_results_and_sets_timeout (test_arxiv_search.ArxivSearchTests.test_real_request_shape_limits_results_and_sets_timeout) ... ok


test_retries_stop_after_three_attempts (test_arxiv_search.ArxivSearchTests.test_retries_stop_after_three_attempts) ... WARNING:root:arXiv request failed: attempt=1/3 error=<urlopen error Synthetic connection failure>


... (23 outputs left out)

----------------------------------------------------------------------


Ran 22 tests in 0.353s


OK


Test process exit code: 0


## Current limits

OCR, image/table interpretation, embeddings, database storage, document search, answer generation, and AWS deployment are not implemented. Parser sorting approximates reading order; extraction quality requires a separate content check. A parser-repaired PDF is flagged. Public availability cannot be inferred solely from arXiv submission timestamps.

Each future development step extends this same notebook after its function and tests are implemented.